# Plan 4 Fisher-risk diagnostic screen

This notebook reads the immutable Phase 4 score-only diagnostic artifact. It performs no learner optimization, score calculation, Fisher estimation, or artifact repair. The screen asks whether a predictable Fisher metric creates actionable adaptive $\pi_t$ signal on paths that defeated the Euclidean controller.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'mnist_experiment':
    ROOT = ROOT.parent
analysis_root = ROOT / 'cache' / 'mnist_experiment' / 'plan4' / 'fisher_analysis'
artifacts = []
for path in analysis_root.glob('phase4__*') if analysis_root.exists() else []:
    if not (path / 'COMPLETED').is_file():
        continue
    candidate = json.loads((path / 'summary.json').read_text())
    if candidate.get('schema_version') == 1:
        artifacts.append((path, candidate))
if not artifacts:
    raise FileNotFoundError('Run run_plan4_fisher_diagnostic; no completed schema-v1 artifact exists.')
ARTIFACT, SUMMARY = max(artifacts, key=lambda item: (item[1]['source_count'], item[0].stat().st_mtime_ns))
print(f'Loaded {ARTIFACT.relative_to(ROOT)}')
print(f"Decision: {SUMMARY['decision']} from {SUMMARY['source_count']} source trajectories")

## Gate summary

`Operational` is the predictable controller after cold-start and bounds. `Ungated` removes cold-start but still uses lagged trend and Fisher state. `Contemporaneous` is deliberately self-coupled to the current score Fisher and is diagnostic only.

In [ ]:
gate_rows = []
for schedule, result in SUMMARY['schedules'].items():
    for half_life, values in result['half_lives'].items():
        metric = values['metrics']
        gate_rows.append({
            'schedule': schedule,
            'half-life': half_life,
            'max operational pi': metric['maximum_operational_pi']['mean'],
            'max ungated pi': metric['maximum_ungated_plugin_pi']['mean'],
            'max contemporaneous pi': metric['maximum_contemporaneous_pi']['mean'],
            'event operational range': metric['operational_event_pi_range']['mean'],
            'event ungated range': metric['ungated_event_pi_range']['mean'],
            'risk lift vs fixed .05': metric['event_relative_operational_risk_reduction_vs_fixed_005']['mean'],
            'Euclidean oracle max': metric['event_euclidean_oracle_maximum_pi']['mean'],
            'cold-start fraction': metric['event_cold_start_fraction']['mean'],
            'pass': values['all_sources_pass_signal_gate'],
        })
gate_table = pd.DataFrame(gate_rows).set_index(['schedule', 'half-life'])
display(gate_table.style.format({
    'max operational pi': '{:.4f}',
    'max ungated pi': '{:.4f}',
    'max contemporaneous pi': '{:.4f}',
    'event operational range': '{:.4f}',
    'event ungated range': '{:.4f}',
    'risk lift vs fixed .05': '{:.2%}',
    'Euclidean oracle max': '{:.4f}',
    'cold-start fraction': '{:.1%}',
}))

## Predictable and self-coupled actions

In [ ]:
colors = {'linear': '#356a8a', 'logistic-k32': '#2f855a', 'logistic-k64': '#b56a22', 'logistic-k128': '#7b5aa6', 'logistic-k256': '#b23a48'}
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for schedule, result in SUMMARY['schedules'].items():
    rows = result['half_lives']['h=0.05']['mean_trajectory']
    p = [row['p'] for row in rows]
    axes[0].plot(p, [row['operational_pi'] for row in rows], label=schedule, color=colors[schedule])
    axes[1].plot(p, [row['ungated_plugin_pi'] for row in rows], label=schedule, color=colors[schedule])
    axes[2].plot(p, [row['contemporaneous_pi'] for row in rows], label=schedule, color=colors[schedule])
for ax, title in zip(axes, ['Operational predictable', 'Ungated predictable', 'Self-coupled contemporaneous']):
    ax.axhline(.05, color='#222222', linestyle='--', linewidth=1, label=r'$\pi_{min}=.05$')
    ax.axhline(.07, color='#777777', linestyle=':', linewidth=1, label='signal gate')
    ax.set(xlabel='$p_t$', ylabel=r'$\pi_t$', title=title, ylim=(0, .10))
axes[2].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
plt.show()

## Fisher-risk numerator and uncertainty

The selected screening schedule is shown at the primary $h=.05$. Movement is $d\theta_t^T G_t d\theta_t$; the old and new covariance terms use the same predictable $G_t$.

In [ ]:
selected = SUMMARY['screening_schedule']
rows = SUMMARY['schedules'][selected]['half_lives']['h=0.05']['mean_trajectory'][:-1]
p = np.asarray([row['p'] for row in rows])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(p, [row['predictable_signal_energy'] for row in rows], label='lagged trend energy', color='#356a8a')
axes[0].plot(p, [row['current_displacement_energy'] for row in rows], label='reference displacement energy', color='#b23a48')
axes[0].set(xlabel='$p_t$', ylabel='Fisher energy', title=f'{selected}: movement numerator')
axes[0].legend(frameon=False)
axes[1].plot(p, [row['old_covariance_risk'] for row in rows], label='old covariance risk', color='#2f855a')
axes[1].plot(p, [row['new_covariance_risk'] for row in rows], label='new covariance risk', color='#b56a22')
axes[1].set(xlabel='$p_t$', ylabel='Fisher risk', title=f'{selected}: uncertainty denominator')
axes[1].legend(frameon=False)
fig.tight_layout()
plt.show()

## Euclidean control and Fisher scale

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for schedule, result in SUMMARY['schedules'].items():
    rows = result['half_lives']['h=0.05']['mean_trajectory'][:-1]
    axes[0].plot([row['p'] for row in rows], [row['euclidean_contemporaneous_oracle_pi'] for row in rows], label=schedule, color=colors[schedule])
    axes[1].plot([row['p'] for row in rows], [row['contemporaneous_pi'] for row in rows], label=schedule, color=colors[schedule])
for ax, title in zip(axes, ['Euclidean design oracle', 'Contemporaneous Fisher oracle']):
    ax.axhline(.05, color='#222222', linestyle='--', linewidth=1)
    ax.axhline(.07, color='#777777', linestyle=':', linewidth=1)
    ax.set(xlabel='$p_t$', ylabel=r'Raw $\pi_t$', title=title, ylim=(0, .10))
axes[1].legend(frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
plt.show()

In [ ]:
message = (
    f"**Decision: {SUMMARY['decision']}.** Across {SUMMARY['source_count']} paired source trajectories, "
    "the predictable Fisher-risk controller remains at the .05 lower bound. "
    "The ungated trajectory does not reveal a hidden event response, and the current-batch "
    "diagnostic does not produce coherent shock-specific actuation."
)
display(Markdown(message))